# 01 - Dataset Audit

This notebook audits the datasets used in the residential building defect
reliability benchmark.

## Target classes

0. biological_growth
1. stain
2. crack
3. peeling
4. spalling

## Dataset components

1. Original public dataset: 3,204 images
2. Additional public dataset: 662 images
3. Independently collected field dataset: 562 images

Background/no-defect images are defined as images with either:
- no YOLO annotation file, or
- an empty YOLO annotation file.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


Import libraries

In [3]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import os
import json

Reproducibility information

In [4]:
import sys
import platform

print("Python version :", sys.version)
print("Platform       :", platform.platform())
print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

Python version : 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Platform       : Linux-6.6.122+-x86_64-with-glibc2.35
Pandas version : 2.2.3
NumPy version  : 2.1.3


Define classes

In [5]:
CLASS_NAMES = {
    0: "biological_growth",
    1: "stain",
    2: "crack",
    3: "peeling",
    4: "spalling"
}

NUM_CLASSES = len(CLASS_NAMES)

print("Number of classes:", NUM_CLASSES)

for class_id, class_name in CLASS_NAMES.items():
    print(class_id, "->", class_name)

Number of classes: 5
0 -> biological_growth
1 -> stain
2 -> crack
3 -> peeling
4 -> spalling


Define dataset paths

In [6]:
# ============================================================
# ORIGINAL PUBLIC DATASET
# ============================================================

OLD_PUBLIC_IMAGES = Path(
    "/content/drive/MyDrive/YOLOV26_1/images"
)

OLD_PUBLIC_LABELS = Path(
    "/content/drive/MyDrive/YOLOV26_1/labels"
)


# ============================================================
# ADDITIONAL PUBLIC DATASET
# ============================================================

NEW_PUBLIC_IMAGES = Path(
    "/content/drive/MyDrive/Paper Dataset/New Dataset/New Images"
)

NEW_PUBLIC_LABELS = Path(
    "/content/drive/MyDrive/Paper Dataset/New Dataset/New Labels"
)


# ============================================================
# INDEPENDENT FIELD DATASET
# ============================================================

FIELD_IMAGES = Path(
    "/content/drive/MyDrive/Paper Dataset/New Dataset/My Dataset/Images"
)

FIELD_LABELS = Path(
    "/content/drive/MyDrive/Paper Dataset/New Dataset/My Dataset/Labels"
)

Check that every folder exists

In [7]:
DATASET_PATHS = {
    "Original Public Images": OLD_PUBLIC_IMAGES,
    "Original Public Labels": OLD_PUBLIC_LABELS,

    "Additional Public Images": NEW_PUBLIC_IMAGES,
    "Additional Public Labels": NEW_PUBLIC_LABELS,

    "Field Images": FIELD_IMAGES,
    "Field Labels": FIELD_LABELS,
}

print("=" * 80)
print("DATASET PATH VERIFICATION")
print("=" * 80)

for name, path in DATASET_PATHS.items():

    status = "OK" if path.exists() else "MISSING"

    print(
        f"{name:<30} : "
        f"{status:<8} "
        f"{path}"
    )

DATASET PATH VERIFICATION
Original Public Images         : OK       /content/drive/MyDrive/YOLOV26_1/images
Original Public Labels         : OK       /content/drive/MyDrive/YOLOV26_1/labels
Additional Public Images       : OK       /content/drive/MyDrive/Paper Dataset/New Dataset/New Images
Additional Public Labels       : OK       /content/drive/MyDrive/Paper Dataset/New Dataset/New Labels
Field Images                   : OK       /content/drive/MyDrive/Paper Dataset/New Dataset/My Dataset/Images
Field Labels                   : OK       /content/drive/MyDrive/Paper Dataset/New Dataset/My Dataset/Labels


Define supported image formats

In [8]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}

### 8. YOLO Dataset Audit Function

The function performs image-level and annotation-level quality control.

It checks:

- total image count
- total label count
- missing annotations
- empty annotations
- background/no-defect images
- class distribution
- instance distribution
- multi-class images
- orphan label files
- malformed YOLO rows
- invalid class IDs
- invalid normalized bounding boxes

In [9]:
def audit_yolo_dataset(
    dataset_name,
    images_dir,
    labels_dir,
    class_names
):

    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)

    # --------------------------------------------------------
    # Find files
    # --------------------------------------------------------

    image_files = sorted([
        p for p in images_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ])

    label_files = sorted([
        p for p in labels_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() == ".txt"
    ])

    image_stems = {p.stem for p in image_files}

    # --------------------------------------------------------
    # Storage
    # --------------------------------------------------------

    class_image_counts = Counter()
    class_instance_counts = Counter()

    missing_labels = []
    empty_labels = []
    background_images = []

    multi_class_images = []

    malformed_rows = []
    invalid_class_rows = []
    invalid_bbox_rows = []

    image_records = []

    # --------------------------------------------------------
    # Process images
    # --------------------------------------------------------

    for image_path in image_files:

        stem = image_path.stem

        label_path = labels_dir / f"{stem}.txt"

        classes_here = []
        valid_instances = 0

        # ----------------------------------------------------
        # Missing label = background
        # ----------------------------------------------------

        if not label_path.exists():

            missing_labels.append(image_path.name)
            background_images.append(image_path.name)

            image_records.append({
                "dataset": dataset_name,
                "image": image_path.name,
                "label_exists": False,
                "label_empty": True,
                "background": True,
                "num_instances": 0,
                "classes": ""
            })

            continue

        # ----------------------------------------------------
        # Empty label = background
        # ----------------------------------------------------

        text = label_path.read_text(
            errors="ignore"
        ).strip()

        if text == "":

            empty_labels.append(label_path.name)
            background_images.append(image_path.name)

            image_records.append({
                "dataset": dataset_name,
                "image": image_path.name,
                "label_exists": True,
                "label_empty": True,
                "background": True,
                "num_instances": 0,
                "classes": ""
            })

            continue

        # ----------------------------------------------------
        # Process annotation rows
        # ----------------------------------------------------

        lines = text.splitlines()

        for line_number, line in enumerate(
            lines,
            start=1
        ):

            parts = line.strip().split()

            # YOLO detection format:
            # class x_center y_center width height

            if len(parts) != 5:

                malformed_rows.append({
                    "dataset": dataset_name,
                    "label": label_path.name,
                    "line": line_number,
                    "content": line
                })

                continue

            try:

                class_id = int(float(parts[0]))

                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])

            except ValueError:

                malformed_rows.append({
                    "dataset": dataset_name,
                    "label": label_path.name,
                    "line": line_number,
                    "content": line
                })

                continue

            # ------------------------------------------------
            # Validate class
            # ------------------------------------------------

            if class_id not in class_names:

                invalid_class_rows.append({
                    "dataset": dataset_name,
                    "label": label_path.name,
                    "line": line_number,
                    "class_id": class_id
                })

                continue

            # ------------------------------------------------
            # Validate YOLO bbox
            # ------------------------------------------------

            if not (
                0 <= x_center <= 1
                and 0 <= y_center <= 1
                and 0 < width <= 1
                and 0 < height <= 1
            ):

                invalid_bbox_rows.append({
                    "dataset": dataset_name,
                    "label": label_path.name,
                    "line": line_number,
                    "class_id": class_id,
                    "x_center": x_center,
                    "y_center": y_center,
                    "width": width,
                    "height": height
                })

                continue

            valid_instances += 1

            classes_here.append(class_id)

            class_instance_counts[class_id] += 1

        # ----------------------------------------------------
        # Image-level class counts
        # ----------------------------------------------------

        unique_classes = sorted(
            set(classes_here)
        )

        for class_id in unique_classes:
            class_image_counts[class_id] += 1

        if len(unique_classes) > 1:
            multi_class_images.append(
                image_path.name
            )

        is_background = valid_instances == 0

        if is_background:
            background_images.append(
                image_path.name
            )

        image_records.append({
            "dataset": dataset_name,
            "image": image_path.name,
            "label_exists": True,
            "label_empty": False,
            "background": is_background,
            "num_instances": valid_instances,
            "classes": ", ".join(
                class_names[c]
                for c in unique_classes
            )
        })

    # --------------------------------------------------------
    # Orphan labels
    # --------------------------------------------------------

    orphan_labels = sorted([
        p.name
        for p in label_files
        if p.stem not in image_stems
    ])

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    total_images = len(image_files)

    total_background = len(
        set(background_images)
    )

    annotated_images = (
        total_images -
        total_background
    )

    background_percentage = (
        total_background /
        total_images *
        100
        if total_images > 0
        else 0
    )

    summary = {
        "dataset": dataset_name,
        "total_images": total_images,
        "total_label_files": len(label_files),
        "annotated_defect_images": annotated_images,
        "background_images": total_background,
        "background_percent": background_percentage,
        "missing_labels": len(missing_labels),
        "empty_labels": len(empty_labels),
        "multi_class_images": len(multi_class_images),
        "orphan_labels": len(orphan_labels),
        "malformed_rows": len(malformed_rows),
        "invalid_class_rows": len(invalid_class_rows),
        "invalid_bbox_rows": len(invalid_bbox_rows)
    }

    # --------------------------------------------------------
    # Class summary
    # --------------------------------------------------------

    class_summary = []

    for class_id, class_name in class_names.items():

        class_summary.append({
            "dataset": dataset_name,
            "class_id": class_id,
            "class_name": class_name,
            "images_with_class":
                class_image_counts[class_id],
            "instances":
                class_instance_counts[class_id]
        })

    # --------------------------------------------------------
    # Return everything
    # --------------------------------------------------------

    return {
        "summary": summary,
        "class_summary":
            pd.DataFrame(class_summary),

        "image_records":
            pd.DataFrame(image_records),

        "missing_labels":
            missing_labels,

        "empty_labels":
            empty_labels,

        "background_images":
            sorted(set(background_images)),

        "multi_class_images":
            multi_class_images,

        "orphan_labels":
            orphan_labels,

        "malformed_rows":
            malformed_rows,

        "invalid_class_rows":
            invalid_class_rows,

        "invalid_bbox_rows":
            invalid_bbox_rows
    }

Audit the original public dataset

In [10]:
old_public_audit = audit_yolo_dataset(
    dataset_name="Original Public",
    images_dir=OLD_PUBLIC_IMAGES,
    labels_dir=OLD_PUBLIC_LABELS,
    class_names=CLASS_NAMES
)

pd.DataFrame(
    [old_public_audit["summary"]]
)

,dataset,total_images,total_label_files,annotated_defect_images,background_images,background_percent,missing_labels,empty_labels,multi_class_images,orphan_labels,malformed_rows,invalid_class_rows,invalid_bbox_rows
0,Original Public,3204,3204,2955,249,7.771536,0,249,122,0,0,0,0


In [12]:
old_public_audit["class_summary"]

,dataset,class_id,class_name,images_with_class,instances
0,Original Public,0,biological_growth,615,679
1,Original Public,1,stain,546,642
2,Original Public,2,crack,850,910
3,Original Public,3,peeling,573,741
4,Original Public,4,spalling,498,528


Audit additional public dataset

In [13]:
new_public_audit = audit_yolo_dataset(
    dataset_name="Additional Public",
    images_dir=NEW_PUBLIC_IMAGES,
    labels_dir=NEW_PUBLIC_LABELS,
    class_names=CLASS_NAMES
)

pd.DataFrame(
    [new_public_audit["summary"]]
)

,dataset,total_images,total_label_files,annotated_defect_images,background_images,background_percent,missing_labels,empty_labels,multi_class_images,orphan_labels,malformed_rows,invalid_class_rows,invalid_bbox_rows
0,Additional Public,662,153,153,509,76.888218,509,0,12,0,0,0,0


In [14]:
new_public_audit["class_summary"]

,dataset,class_id,class_name,images_with_class,instances
0,Additional Public,0,biological_growth,0,0
1,Additional Public,1,stain,18,33
2,Additional Public,2,crack,70,82
3,Additional Public,3,peeling,27,30
4,Additional Public,4,spalling,50,66


Audit independent field dataset

In [15]:
field_audit = audit_yolo_dataset(
    dataset_name="External Field",
    images_dir=FIELD_IMAGES,
    labels_dir=FIELD_LABELS,
    class_names=CLASS_NAMES
)

pd.DataFrame(
    [field_audit["summary"]]
)

,dataset,total_images,total_label_files,annotated_defect_images,background_images,background_percent,missing_labels,empty_labels,multi_class_images,orphan_labels,malformed_rows,invalid_class_rows,invalid_bbox_rows
0,External Field,562,158,158,404,71.886121,404,0,32,0,0,0,0


In [16]:
field_audit["class_summary"]

,dataset,class_id,class_name,images_with_class,instances
0,External Field,0,biological_growth,24,27
1,External Field,1,stain,82,97
2,External Field,2,crack,38,49
3,External Field,3,peeling,34,46
4,External Field,4,spalling,15,16


Compare all three datasets

In [17]:
dataset_summary_df = pd.DataFrame([
    old_public_audit["summary"],
    new_public_audit["summary"],
    field_audit["summary"]
])

dataset_summary_df

,dataset,total_images,total_label_files,annotated_defect_images,background_images,background_percent,missing_labels,empty_labels,multi_class_images,orphan_labels,malformed_rows,invalid_class_rows,invalid_bbox_rows
0,Original Public,3204,3204,2955,249,7.771536,0,249,122,0,0,0,0
1,Additional Public,662,153,153,509,76.888218,509,0,12,0,0,0,0
2,External Field,562,158,158,404,71.886121,404,0,32,0,0,0,0


Calculate total public development dataset

In [18]:
public_total_images = (
    old_public_audit["summary"]["total_images"]
    +
    new_public_audit["summary"]["total_images"]
)

public_defect_images = (
    old_public_audit["summary"]["annotated_defect_images"]
    +
    new_public_audit["summary"]["annotated_defect_images"]
)

public_background_images = (
    old_public_audit["summary"]["background_images"]
    +
    new_public_audit["summary"]["background_images"]
)

print("=" * 60)
print("PUBLIC DEVELOPMENT CORPUS")
print("=" * 60)

print(
    "Total images      :",
    public_total_images
)

print(
    "Defect images     :",
    public_defect_images
)

print(
    "Background images :",
    public_background_images
)

print(
    "Background %      :",
    f"{public_background_images/public_total_images*100:.2f}%"
)

PUBLIC DEVELOPMENT CORPUS
Total images      : 3866
Defect images     : 3108
Background images : 758
Background %      : 19.61%


Calculate total study dataset

In [19]:
grand_total = (
    public_total_images
    +
    field_audit["summary"]["total_images"]
)

grand_defect = (
    public_defect_images
    +
    field_audit["summary"]["annotated_defect_images"]
)

grand_background = (
    public_background_images
    +
    field_audit["summary"]["background_images"]
)

print("=" * 60)
print("COMPLETE DATASET")
print("=" * 60)

print("Total images      :", grand_total)
print("Defect images     :", grand_defect)
print("Background images :", grand_background)

COMPLETE DATASET
Total images      : 4428
Defect images     : 3266
Background images : 1162


Combined public class distribution

In [20]:
old_classes = (
    old_public_audit["class_summary"]
    .set_index("class_id")
)

new_classes = (
    new_public_audit["class_summary"]
    .set_index("class_id")
)

combined_public_classes = pd.DataFrame({
    "class_name":
        old_classes["class_name"],

    "images_with_class":
        old_classes["images_with_class"]
        +
        new_classes["images_with_class"],

    "instances":
        old_classes["instances"]
        +
        new_classes["instances"]
})

combined_public_classes

,class_name,images_with_class,instances
class_id,,,
0,biological_growth,615,679
1,stain,564,675
2,crack,920,992
3,peeling,600,771
4,spalling,548,594


Field class distribution

In [21]:
field_class_table = (
    field_audit["class_summary"][
        [
            "class_name",
            "images_with_class",
            "instances"
        ]
    ]
)

field_class_table

,class_name,images_with_class,instances
0,biological_growth,24,27
1,stain,82,97
2,crack,38,49
3,peeling,34,46
4,spalling,15,16


Check annotation quality across all datasets

In [22]:
quality_columns = [
    "dataset",
    "missing_labels",
    "empty_labels",
    "orphan_labels",
    "malformed_rows",
    "invalid_class_rows",
    "invalid_bbox_rows"
]

quality_summary = dataset_summary_df[
    quality_columns
]

quality_summary

,dataset,missing_labels,empty_labels,orphan_labels,malformed_rows,invalid_class_rows,invalid_bbox_rows
0,Original Public,0,249,0,0,0,0
1,Additional Public,509,0,0,0,0,0
2,External Field,404,0,0,0,0,0


Check filename overlap between datasets

In [23]:
def image_filename_set(folder):

    return {
        p.name
        for p in Path(folder).iterdir()
        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    }


old_names = image_filename_set(
    OLD_PUBLIC_IMAGES
)

new_names = image_filename_set(
    NEW_PUBLIC_IMAGES
)

field_names = image_filename_set(
    FIELD_IMAGES
)


old_new_overlap = (
    old_names & new_names
)

old_field_overlap = (
    old_names & field_names
)

new_field_overlap = (
    new_names & field_names
)


print(
    "Original public vs additional public:",
    len(old_new_overlap)
)

print(
    "Original public vs field:",
    len(old_field_overlap)
)

print(
    "Additional public vs field:",
    len(new_field_overlap)
)

Original public vs additional public: 0
Original public vs field: 0
Additional public vs field: 0


Check duplicate label filenames

In [24]:
def label_filename_set(folder):

    return {
        p.name
        for p in Path(folder).iterdir()
        if p.is_file()
        and p.suffix.lower() == ".txt"
    }


old_label_names = label_filename_set(
    OLD_PUBLIC_LABELS
)

new_label_names = label_filename_set(
    NEW_PUBLIC_LABELS
)

field_label_names = label_filename_set(
    FIELD_LABELS
)


print(
    "Old vs new label-name overlap:",
    len(old_label_names & new_label_names)
)

print(
    "Old vs field label-name overlap:",
    len(old_label_names & field_label_names)
)

print(
    "New vs field label-name overlap:",
    len(new_label_names & field_label_names)
)

Old vs new label-name overlap: 0
Old vs field label-name overlap: 0
New vs field label-name overlap: 0


Inspect multi-class-image frequency

In [25]:
multi_class_summary = pd.DataFrame({
    "dataset": [
        "Original Public",
        "Additional Public",
        "External Field"
    ],

    "multi_class_images": [
        len(
            old_public_audit[
                "multi_class_images"
            ]
        ),

        len(
            new_public_audit[
                "multi_class_images"
            ]
        ),

        len(
            field_audit[
                "multi_class_images"
            ]
        )
    ]
})

multi_class_summary

,dataset,multi_class_images
0,Original Public,122
1,Additional Public,12
2,External Field,32


Check class imbalance

In [26]:
class_imbalance = (
    combined_public_classes.copy()
)

max_instances = (
    class_imbalance["instances"].max()
)

class_imbalance[
    "relative_to_largest_class"
] = (
    class_imbalance["instances"]
    /
    max_instances
)

class_imbalance

,class_name,images_with_class,instances,relative_to_largest_class
class_id,,,,
0,biological_growth,615,679,0.684476
1,stain,564,675,0.680444
2,crack,920,992,1.000000
3,peeling,600,771,0.777218
4,spalling,548,594,0.598790


Save all audit tables

In [27]:
AUDIT_OUTPUT = Path(
    "/content/drive/MyDrive/Paper Dataset/"
    "Research_Dataset_Audit"
)

AUDIT_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

dataset_summary_df.to_csv(
    AUDIT_OUTPUT /
    "dataset_summary.csv",
    index=False
)

combined_public_classes.to_csv(
    AUDIT_OUTPUT /
    "public_class_distribution.csv"
)

field_class_table.to_csv(
    AUDIT_OUTPUT /
    "field_class_distribution.csv",
    index=False
)

quality_summary.to_csv(
    AUDIT_OUTPUT /
    "annotation_quality_summary.csv",
    index=False
)

multi_class_summary.to_csv(
    AUDIT_OUTPUT /
    "multi_class_summary.csv",
    index=False
)

print(
    "Audit results saved to:",
    AUDIT_OUTPUT
)

Audit results saved to: /content/drive/MyDrive/Paper Dataset/Research_Dataset_Audit


Final automated integrity check

In [28]:
critical_errors = 0

for audit_name, audit in [
    ("Original Public", old_public_audit),
    ("Additional Public", new_public_audit),
    ("External Field", field_audit)
]:

    critical_errors += len(
        audit["orphan_labels"]
    )

    critical_errors += len(
        audit["malformed_rows"]
    )

    critical_errors += len(
        audit["invalid_class_rows"]
    )

    critical_errors += len(
        audit["invalid_bbox_rows"]
    )


print("=" * 70)
print("FINAL DATASET INTEGRITY CHECK")
print("=" * 70)

print(
    "Complete dataset images :",
    grand_total
)

print(
    "Public development pool :",
    public_total_images
)

print(
    "External field dataset  :",
    field_audit["summary"]["total_images"]
)

print(
    "Critical annotation errors:",
    critical_errors
)

if critical_errors == 0:

    print(
        "\n✅ BASIC DATASET INTEGRITY CHECK PASSED"
    )

else:

    print(
        "\n❌ DATASET REQUIRES CORRECTION"
    )

FINAL DATASET INTEGRITY CHECK
Complete dataset images : 4428
Public development pool : 3866
External field dataset  : 562
Critical annotation errors: 0

✅ BASIC DATASET INTEGRITY CHECK PASSED


## 24. Dataset Audit Conclusion

The dataset audit confirmed:

- 4,428 total images were available.
- 3,866 public-source images were retained for model development.
- 562 independently collected field images were reserved for external validation.
- The public development corpus contained both defect and no-defect/background images.
- The external field dataset also contained both defect and no-defect observations.
- All five target defect classes were represented in the external field dataset.
- No malformed annotation rows, invalid class identifiers, invalid bounding boxes or orphan labels were detected in the audited datasets.

The next stage is duplicate and near-duplicate analysis of the public development corpus before the final train/validation/internal-test split is generated.